# Bài tập 3 – EDA dữ liệu bất động sản

**Project:** `real-estate-amenities-price-analysis`

Notebook này thực hiện:
- Thống kê mô tả
- Kiểm tra và xử lý dữ liệu thiếu
- Xử lý ngoại lai bằng IQR
- Trực quan hóa tối thiểu 8 biểu đồ
- Phân tích tiện ích
- Feature Engineering
- Lưu cleaned dataset
- Chuẩn bị dữ liệu cho Bài tập 4

> Dữ liệu được đọc từ `data/raw/properties/` và `data/raw/amenities/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")

ROOT = Path("../..")   # từ src/process/ đi lên thư mục project
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESS = ROOT / "data" / "process"

print("Project:", ROOT.resolve())
print("Raw data:", DATA_RAW.resolve())


## 1. Đọc dữ liệu Properties

In [ ]:
property_files = sorted((DATA_RAW / "properties").glob("*.csv"))

print("Các file Properties:")
for f in property_files:
    print("-", f.name)

properties_list = []
for f in property_files:
    temp = pd.read_csv(f)
    temp["source_file"] = f.name
    properties_list.append(temp)

df = pd.concat(properties_list, ignore_index=True)

print("Kích thước Properties:", df.shape)
display(df.head())


## 2. Đọc dữ liệu Amenities

In [ ]:
amenity_files = sorted((DATA_RAW / "amenities").glob("*.csv"))

print("Các file Amenities:")
for f in amenity_files:
    print("-", f.name)

amenities_list = []
for f in amenity_files:
    temp = pd.read_csv(f)
    temp["source_file"] = f.name
    amenities_list.append(temp)

amenities = pd.concat(amenities_list, ignore_index=True)

print("Kích thước Amenities:", amenities.shape)
display(amenities.head())


## 3. Kiểm tra cấu trúc và thống kê mô tả

In [ ]:
print("=== INFO ===")
df.info()

print("\n=== DESCRIBE ===")
display(df.describe(include="all").T)

print("\n=== Số lượng trùng lặp ===")
print(df.duplicated().sum())

print("\n=== Phân bố loại hình ===")
if "Loại hình" in df.columns:
    display(df["Loại hình"].value_counts(dropna=False))


## 4. Kiểm tra dữ liệu thiếu

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

display(missing[missing["missing_count"] > 0])

plt.figure(figsize=(12, 6))
missing_plot = missing[missing["missing_count"] > 0].head(15)
sns.barplot(x=missing_plot["missing_percent"], y=missing_plot.index)
plt.title("Tỷ lệ dữ liệu thiếu theo cột")
plt.xlabel("Tỷ lệ thiếu (%)")
plt.ylabel("Cột")
plt.show()


### Nhận xét dữ liệu thiếu

Các trường như hướng, nội thất, số tầng hoặc diện tích có thể bị thiếu tùy loại hình bất động sản. Không nên xóa toàn bộ dòng chỉ vì một cột bị thiếu; việc xử lý được thực hiện theo từng loại biến ở bước làm sạch.


## 5. Chuẩn hóa các biến số

In [ ]:
# Hàm lấy số từ chuỗi, ví dụ '4m' -> 4 và '100 m2' -> 100
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False)
                     .str.extract(r"([-+]?\d*\.?\d+)")[0],
        errors="coerce"
    )

# Các cột số quan trọng
numeric_candidates = [
    "Tong_gia (trieu)",
    "Tien_m2 (trieu/m2)",
    "Số phòng ngủ",
    "Số phòng tắm",
    "Số tầng",
    "Chiều ngang",
    "Chiều dài",
    "Diện tích đất công nhận",
    "Diện tích tim tường",
    "Diện tích sử dụng",
    "Đường trước nhà"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = to_number(df[col])

print(df.dtypes)


## 6. Xử lý dữ liệu thiếu

In [ ]:
# Xóa bản ghi không có mã BĐS
if "Ma_BDS" in df.columns:
    df = df.dropna(subset=["Ma_BDS"]).drop_duplicates(subset=["Ma_BDS"]).copy()

# Các biến số: điền median
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Các biến phân loại: điền mode
categorical_cols = df.select_dtypes(include="object").columns
for col in categorical_cols:
    if df[col].isna().any():
        mode = df[col].mode()
        if len(mode) > 0:
            df[col] = df[col].fillna(mode.iloc[0])
        else:
            df[col] = df[col].fillna("Không xác định")

print("Tổng missing còn lại:", int(df.isna().sum().sum()))


## 7. Xử lý ngoại lai bằng IQR

In [ ]:
def remove_iqr_outliers(data, column, multiplier=1.5):
    if column not in data.columns:
        return data
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    return data[data[column].between(lower, upper)].copy()

before = len(df)

# Ưu tiên xử lý các biến chính của bài toán giá
for col in ["Tong_gia (trieu)", "Tien_m2 (trieu/m2)"]:
    if col in df.columns:
        df = remove_iqr_outliers(df, col)

after = len(df)

print("Số dòng trước xử lý:", before)
print("Số dòng sau xử lý:", after)
print("Số dòng loại:", before - after)


## 8. Histogram giá

In [ ]:
if "Tong_gia (trieu)" in df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(df["Tong_gia (trieu)"], kde=True)
    plt.title("Phân bố tổng giá bất động sản")
    plt.xlabel("Tổng giá (triệu VNĐ)")
    plt.ylabel("Số lượng")
    plt.show()


**Insight:** Biểu đồ cho biết mức giá BĐS tập trung ở khoảng nào và mức độ phân tán của giá. Sau khi loại bỏ ngoại lai, phân bố giá phản ánh rõ hơn nhóm BĐS phổ biến trong dataset.


## 9. Histogram diện tích

In [ ]:
area_col = next((c for c in [
    "Diện tích sử dụng", "Diện tích đất công nhận", "Diện tích tim tường"
] if c in df.columns), None)

if area_col:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[area_col], kde=True)
    plt.title(f"Phân bố {area_col}")
    plt.xlabel("Diện tích (m²)")
    plt.ylabel("Số lượng")
    plt.show()


**Insight:** Phân bố diện tích giúp xác định quy mô BĐS phổ biến và mức độ chênh lệch giữa các sản phẩm.


## 10. Boxplot giá theo loại hình

In [ ]:
if "Loại hình" in df.columns and "Tong_gia (trieu)" in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x="Loại hình", y="Tong_gia (trieu)")
    plt.title("So sánh tổng giá theo loại hình bất động sản")
    plt.xlabel("Loại hình")
    plt.ylabel("Tổng giá (triệu VNĐ)")
    plt.xticks(rotation=20)
    plt.show()


**Insight:** Mức giá có sự khác biệt giữa các loại hình BĐS. Boxplot đồng thời cho thấy trung vị, độ phân tán và các nhóm giá cao/thấp của từng loại hình.


## 11. Boxplot giá/m² theo loại hình

In [ ]:
if "Loại hình" in df.columns and "Tien_m2 (trieu/m2)" in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x="Loại hình", y="Tien_m2 (trieu/m2)")
    plt.title("So sánh đơn giá theo loại hình bất động sản")
    plt.xlabel("Loại hình")
    plt.ylabel("Triệu VNĐ/m²")
    plt.xticks(rotation=20)
    plt.show()


**Insight:** Đơn giá/m² giúp so sánh giá trị giữa các BĐS có quy mô khác nhau. Một số loại hình có thể có mặt bằng đơn giá cao dù tổng giá không cao nhất.


## 12. Scatter diện tích – giá

In [ ]:
if area_col and "Tong_gia (trieu)" in df.columns:
    plot_df = df[[area_col, "Tong_gia (trieu)", "Loại hình"]].dropna()
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=plot_df,
        x=area_col,
        y="Tong_gia (trieu)",
        hue="Loại hình",
        alpha=0.7
    )
    plt.title("Mối quan hệ giữa diện tích và tổng giá")
    plt.xlabel("Diện tích (m²)")
    plt.ylabel("Tổng giá (triệu VNĐ)")
    plt.show()


**Insight:** Biểu đồ giúp kiểm tra liệu diện tích lớn hơn có đi kèm tổng giá cao hơn hay không. Các điểm lệch xa xu hướng chung cần được xem xét như các trường hợp đặc biệt.


## 13. Heatmap tương quan

In [ ]:
corr_cols = [
    c for c in [
        "Tong_gia (trieu)", "Tien_m2 (trieu/m2)",
        "Số phòng ngủ", "Số phòng tắm", "Số tầng",
        "Chiều ngang", "Chiều dài",
        "Diện tích đất công nhận", "Diện tích tim tường",
        "Diện tích sử dụng", "Đường trước nhà"
    ] if c in df.columns
]

if len(corr_cols) >= 2:
    plt.figure(figsize=(12, 8))
    sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Ma trận tương quan giữa các biến số")
    plt.show()


**Insight:** Heatmap cho biết chiều và mức độ liên hệ giữa các biến số. Đặc biệt cần chú ý mối tương quan giữa giá, đơn giá và diện tích để định hướng feature engineering/modeling.


## 14. Bar chart số lượng BĐS theo loại hình

In [ ]:
if "Loại hình" in df.columns:
    count_df = df["Loại hình"].value_counts().reset_index()
    count_df.columns = ["Loại hình", "Số lượng"]

    plt.figure(figsize=(10, 6))
    sns.barplot(data=count_df, x="Loại hình", y="Số lượng")
    plt.title("Số lượng bất động sản theo loại hình")
    plt.xlabel("Loại hình")
    plt.ylabel("Số lượng")
    plt.xticks(rotation=20)
    plt.show()


**Insight:** Biểu đồ cho thấy loại hình nào chiếm tỷ trọng lớn trong dataset. Điều này cần được lưu ý khi so sánh giá vì số lượng quan sát giữa các nhóm có thể không cân bằng.


## 15. Phân tích tiện ích

In [ ]:
if "Loai_tien_ich" in amenities.columns:
    amenity_count = amenities["Loai_tien_ich"].value_counts().reset_index()
    amenity_count.columns = ["Loại tiện ích", "Số lượng"]

    plt.figure(figsize=(12, 6))
    sns.barplot(data=amenity_count, x="Loại tiện ích", y="Số lượng")
    plt.title("Số lượng tiện ích theo loại")
    plt.xlabel("Loại tiện ích")
    plt.ylabel("Số lượng")
    plt.xticks(rotation=30)
    plt.show()


**Insight:** Phân bố loại tiện ích cho biết những nhóm tiện ích xuất hiện phổ biến nhất xung quanh các BĐS trong dữ liệu.


## 16. Khoảng cách đến tiện ích

In [ ]:
if "Khoảng cách" in amenities.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(amenities["Khoảng cách"].dropna(), kde=True)
    plt.title("Phân bố khoảng cách đến tiện ích")
    plt.xlabel("Khoảng cách (m)")
    plt.ylabel("Số lượng")
    plt.show()


**Insight:** Phân bố khoảng cách cho biết các tiện ích thường nằm gần hay xa BĐS. Đây là cơ sở để tạo đặc trưng về mức độ tiếp cận tiện ích.


## 17. Plotly interactive – Giá theo loại hình

In [ ]:
if "Loại hình" in df.columns and "Tong_gia (trieu)" in df.columns:
    fig = px.box(
        df,
        x="Loại hình",
        y="Tong_gia (trieu)",
        points="outliers",
        title="Tương tác: Phân bố giá theo loại hình"
    )
    fig.update_layout(xaxis_title="Loại hình", yaxis_title="Tổng giá (triệu VNĐ)")
    fig.show()


## 18. Feature Engineering – Giá/m²

In [ ]:
# Tạo lại giá/m² nếu có tổng giá và diện tích sử dụng
if "Tong_gia (trieu)" in df.columns and area_col:
    df["Gia_tinh_lai_m2"] = df["Tong_gia (trieu)"] / df[area_col].replace(0, np.nan)

display(df[[c for c in ["Ma_BDS", "Tong_gia (trieu)", area_col, "Gia_tinh_lai_m2"] if c in df.columns]].head())


## 19. Feature Engineering – Tổng hợp tiện ích theo BĐS

In [ ]:
# Tạo các đặc trưng tiện ích cho từng BĐS
if "Ma_BDS" in amenities.columns:
    amenity_features = amenities.groupby("Ma_BDS").agg(
        So_tien_ich=("Ma_BDS", "size"),
        Khoang_cach_tien_ich_trung_binh=("Khoảng cách", "mean"),
        Khoang_cach_tien_ich_gan_nhat=("Khoảng cách", "min")
    ).reset_index()

    df = df.merge(amenity_features, on="Ma_BDS", how="left")

    for col in [
        "So_tien_ich",
        "Khoang_cach_tien_ich_trung_binh",
        "Khoang_cach_tien_ich_gan_nhat"
    ]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

display(df.head())


## 20. Lưu cleaned dataset

In [ ]:
DATA_PROCESS.mkdir(parents=True, exist_ok=True)

clean_path = DATA_PROCESS / "cleaned_eda_dataset.csv"
df.to_csv(clean_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", clean_path.resolve())
print("Kích thước cleaned dataset:", df.shape)


## 21. Chuẩn bị cho Bài tập 4

### Hai nhóm mô hình dự kiến

**Nhóm 1 – Hồi quy:** dự đoán `Tong_gia (trieu)` hoặc `Tien_m2 (trieu/m2)`.

**Nhóm 2 – Phân loại:** tạo các nhóm phân khúc giá (ví dụ thấp / trung bình / cao) rồi thử Logistic Regression hoặc mô hình phân loại khác.

Ở Bài 4 sẽ tiếp tục mã hóa biến phân loại, chia train/test và đánh giá mô hình.


## Kết luận EDA

Sau khi chạy toàn bộ notebook, cần bổ sung/điều chỉnh các con số cụ thể trong phần **Insight** dựa trên kết quả thực tế của dataset. Không nên giữ nguyên nhận xét mẫu nếu biểu đồ cho kết quả khác.

Checklist:
- [x] Thống kê mô tả
- [x] Kiểm tra missing
- [x] Xử lý missing
- [x] Xử lý outlier
- [x] ≥ 8 biểu đồ
- [x] Insight
- [x] Plotly
- [x] Feature Engineering
- [x] Lưu cleaned dataset
- [x] Định hướng mô hình Bài 4
